In [2]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"



In [3]:
import my_data_manager as mdm

cat = mdm.load_cat("Data/galaxies_subhalos_zphot.cat")
halos = mdm.separate_halos(cat)
clusters, groups = mdm.separate_clusters(halos)
clusters = clusters[:678]


/home/bored/projects/SubstructureBenchmarking/my_data_manager.py:53: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(cat, delim_whitespace=True, header=None)
/home/bored/projects/SubstructureBenchmarking/my_data_manager.py:53: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(cat, delim_whitespace=True, header=None)


In [4]:
clean_clusters = [df[df.groupby("haloId")["haloId"].transform("count") > 3].reset_index(drop=True) for df in clusters]

In [5]:
import pandas as pd

N = 21

splus_clusters = [df[pd.to_numeric(df['mag_r'], errors = 'coerce') >= N].copy() for df in clusters]

In [6]:
cluster_samples = {
    "Raw": clusters,
    "Clean": clean_clusters,
    "SPLUS": splus_clusters
}

## Clustering

In [7]:
import clustering_methods as clustering
import numpy as np

algorithms = {}


algorithms['BGMM'] = clustering.run_GMM
algorithms['DBSCAN'] = clustering.run_DBSCAN
algorithms['HDBSCAN'] = clustering.run_HDBSCAN
algorithms['Optics'] = clustering.run_OPTICS
algorithms['Kmeans'] = clustering.run_Kmeans
algorithms['Agglomerative'] = clustering.run_Aglomerative_Clustering
algorithms['Affinity'] = clustering.run_Affinity_Propagation

params = {
    "max_clusters" : 0.1,
    "covariance_type" : 'full',
    "min_cluster_size" : 4,
    "min_eps" : 0.5,
    "max_eps" : np.inf,
    "linkage" : 'ward',
    "clustering_threshold" : 0.5,
    "max_iter" : 1000
}

In [ ]:
predictions = []
    # 1. Initialize as a list instead of a scalar
    iteration_times = [] import time

def ml_worker(alg, samples):

    

    for sample in samples:
    
        X_data = sample["RA"].values
        Y_data = sample["DEC"].values
        
        data = np.column_stack((X_data, Y_data))
        data = np.asarray(data, dtype=np.float64)

        start_time = time.perf_counter()
        labels, probs, c = algorithms[alg](data, params)
        end_time = time.perf_counter()
        
        delta_time = end_time - start_time
        # 2. Store the specific time for this iteration
        iteration_times.append(delta_time) 
        
        sample["firstHaloInFOFGroupId"] = (
            sample["firstHaloInFOFGroupId"]
            .astype(str)
        )

        fof_id = sample["firstHaloInFOFGroupId"].iloc[0]
        labels = np.insert(labels.astype(str), 0, fof_id)

        predictions.append(labels)
    
    return iteration_times, predictions

In [9]:
from ds_plus import milaDS
import astro_utils as au
import numpy as np
import time
from astropy.stats import biweight_location


def dsp_worker(samples):

    predictions = []
    # 1. Change from scalar to list
    iteration_times = []

    for sample in samples:
        
        X_data = np.asarray(sample["RA"].values, dtype=np.float64)
        Y_data = np.asarray(sample["DEC"].values, dtype=np.float64)
        Z_data = np.asarray(sample["z_app"].values, dtype=np.float64)
        
        x_cluster = biweight_location(X_data)
        y_cluster = biweight_location(Y_data)
        Z_clus = biweight_location(Z_data)

        X_Kpc, Y_Kpc = au.gal_Mpc_coords(X_data, Y_data, Z_data, x_cluster, y_cluster)
        X_Kpc *= 1000
        Y_Kpc *= 1000
        
        V_data = au.los_vel(Z_data, Z_clus)

        # --- Timing Start ---
        start_time = time.perf_counter()
        galaxy_info, grouping, summary = milaDS.DSp_groups(X_data, Y_data, V_data, Z_clus)
        end_time = time.perf_counter()
        
        delta_time = end_time - start_time
        # 2. Store individual iteration time
        iteration_times.append(delta_time)
        # --- Timing End ---
        
        # Extract the 9th column (group ID)
        labels = np.array([row[8] for row in grouping])

        sample["firstHaloInFOFGroupId"] = (
            sample["firstHaloInFOFGroupId"]
            .astype(str)
        )

        fof_id = sample["firstHaloInFOFGroupId"].iloc[0]
        # Insert the fof_id at the start of the array
        labels = np.insert(labels.astype(str), 0, fof_id)

        predictions.append(labels)

    # 3. Return the list of times
    return iteration_times, predictions

In [10]:
from calsagos import lagasu
from calsagos import utils
from calsagos import clumberi
from astropy.stats import biweight_location
import numpy as np
import time

from IPython.display import clear_output

def calsagos_worker(samples):
    #- S-PLUS mock cosmology
    H_mock = 67.3
    Omega_L_mock = 0.685
    Omega_m_mock = 0.315

    range_cut_percentage = 0.1
    #-- GENERAL PARAMETERS
    n_galaxies = 4 

    predictions = []
    # 1. Initialize as a list to store time for each sample
    iteration_times = [] 

    for sample in samples:
        X_data = np.asarray(sample["RA"].values, dtype=np.float64)
        Y_data = np.asarray(sample["DEC"].values, dtype=np.float64)
        Z_data = np.asarray(sample["z_app"].values, dtype=np.float64)

        x_cluster = biweight_location(X_data)
        y_cluster = biweight_location(Y_data)
        Z_clus = biweight_location(Z_data)
        
        cluster_mass = float(sample["log(m_200)"].iloc[0])
        range_cuts = int(len(sample)*range_cut_percentage)
        
        # --- START TIMING ---
        start_time = time.perf_counter()
        
        id_galaxy = indexes = np.arange(len(sample) + 1)

        r200_kpc = utils.calc_radius_finn(cluster_mass, Z_clus, H_mock, Omega_L_mock, Omega_m_mock, "kiloparsec")

        r200_degree = utils.convert_kpc_to_angular_distance(r200_kpc, Z_clus, H_mock, Omega_m_mock, "degrees") 

        cluster_members = clumberi.clumberi(id_galaxy, X_data, Y_data, Z_data, Z_clus, x_cluster, y_cluster, range_cuts)

        id_member = cluster_members[0]
        ra_member = cluster_members[1]
        dec_member = cluster_members[2]
        redshift_member = cluster_members[3]
        
        knn_distance = utils.calc_knn_galaxy_distance(ra_member, dec_member, n_galaxies)
        
        knn_galaxy_distance = knn_distance[0]

        try:
            typical_separation = utils.best_eps_dbscan(id_member, knn_galaxy_distance)
        except Exception as e:
            clear_output(wait=True)
            print("Error in best_eps_dbscan:", e)
            print(id_member)
            print(len(id_member))
            print(knn_distance)
            print(len(knn_distance))


        label_candidates = lagasu.lagasu(id_galaxy, X_data, Y_data, Z_data, 
                            range_cuts, typical_separation, n_galaxies, 'euclidean', 'dbscan', 
                            x_cluster, y_cluster, Z_clus, 
                            r200_degree, 'zspec')

        # --- END TIMING ---
        end_time = time.perf_counter()
        delta_time = end_time - start_time

        id_candidates = label_candidates[0]
        ra_candidates = label_candidates[1]
        dec_candidates = label_candidates[2]
        redshift_candidates = label_candidates[3]
        label_zcut = label_candidates[4]
        label_final = label_candidates[5]
        
        # 2. Store the specific time for this iteration
        iteration_times.append(delta_time)

        sample["firstHaloInFOFGroupId"] = sample["firstHaloInFOFGroupId"].astype(str)
        fof_id = sample["firstHaloInFOFGroupId"].iloc[0]
        
        # Consistent with your example: prepending the fof_id to labels
        labels_with_id = np.insert(label_final.astype(str), 0, fof_id)
        predictions.append(labels_with_id)

    # 3. Return the list of times instead of the total sum
    return iteration_times, predictions

In [11]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def run_ml_worker(worker, alg, thread_count, iterations, samples):
    results = []
    
    with ProcessPoolExecutor(max_workers=thread_count) as executor:
            futures = [executor.submit(worker, alg, samples) for _ in range(iterations)]
            for future in as_completed(futures):
                results.append(future.result())
    
    return results

In [12]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def run_worker(worker, thread_count, iterations, samples):
    results = []

    with ProcessPoolExecutor(max_workers=thread_count) as executor:
        futures = [executor.submit(worker, samples) for _ in range(iterations)]

        for future in as_completed(futures):
            results.append(future.result())
    
    return results

In [13]:
from openpyxl import Workbook
from itertools import zip_longest

def save_xlsx(results, title):
    wb = Workbook()

    # 1. Setup the Execution Times sheet (Rows = Iterations, Cols = Samples/Runs)
    sheet1 = wb.active
    sheet1.title = "Execution Times"
    
    # Extract just the duration lists from the results
    # results = [( [times], [preds] ), ( [times], [preds] )]
    all_duration_lists = [res[0] for res in results]
    
    # Create Headers: "Iteration", "Sample 1", "Sample 2", etc.
    headers = ["Iteration"] + [f"Run {i+1}" for i in range(len(all_duration_lists))]
    sheet1.append(headers)

    # Use zip_longest to pair up times by iteration index
    # fillvalue="" handles cases where one run has fewer iterations than others
    for idx, row_times in enumerate(zip_longest(*all_duration_lists, fillvalue="")):
        # Append iteration number (idx+1) followed by the times for that iteration
        sheet1.append([idx + 1] + list(row_times))

    # 2. Store the prediction data in separate sheets as before
    for res_idx, (_, data) in enumerate(results):
        sheet = wb.create_sheet(title=f"Result_{res_idx + 1}")
        
        # This keeps your original logic for predictions
        for row in zip_longest(*data, fillvalue=""):
            sheet.append(row)

    wb.save(f"{title}.xlsx")

In [14]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
threads = 5
iterations = 10

for cs in cluster_samples.keys():
    cluster_sample = cluster_samples[cs]

    folder = f"Results/{cs}_samples"
    os.makedirs(folder, exist_ok=True)

    for alg in algorithms.keys():
        ml_results = run_ml_worker(ml_worker, alg, threads, iterations, cluster_sample)
        save_xlsx(ml_results, os.path.join(folder,f"{alg}"))
        print(f"saved to {folder}/{alg}")

    #dsp_results = run_worker(dsp_worker, threads, iterations, cluster_sample)
    #save_xlsx(dsp_results, os.path.join(folder,f"DSP"))
    #print(f"saved to {folder}/DSP")

    #calsagos_results = run_worker(calsagos_worker, threads, iterations, cluster_sample)
    #print(f"saved to {folder}/CALSAGOS")
    #save_xlsx(calsagos_results, os.path.join(folder,f"CALSAGOS"))

saved to Results/Raw_samples/BGMM
saved to Results/Raw_samples/DBSCAN
saved to Results/Raw_samples/HDBSCAN
saved to Results/Raw_samples/Optics
saved to Results/Raw_samples/Kmeans
saved to Results/Raw_samples/Agglomerative
saved to Results/Raw_samples/Affinity
saved to Results/Clean_samples/BGMM
saved to Results/Clean_samples/DBSCAN
saved to Results/Clean_samples/HDBSCAN
saved to Results/Clean_samples/Optics
saved to Results/Clean_samples/Kmeans
saved to Results/Clean_samples/Agglomerative
saved to Results/Clean_samples/Affinity
saved to Results/SPLUS_samples/BGMM
saved to Results/SPLUS_samples/DBSCAN
saved to Results/SPLUS_samples/HDBSCAN
saved to Results/SPLUS_samples/Optics
saved to Results/SPLUS_samples/Kmeans
saved to Results/SPLUS_samples/Agglomerative
saved to Results/SPLUS_samples/Affinity
